<a href="https://colab.research.google.com/github/srayashidas9/accident-prone-road-prediction-kolkata/blob/main/kolkata_fractal_grid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kolkata Road/Building Fractal Grid
Run every cell top to bottom: **Runtime menu -> Run all**.
Takes 2-5 minutes. Output file `kolkata_fractal_grid.csv` will appear in the Files panel (folder icon, left sidebar) when done.

In [1]:
!pip install osmnx geopandas shapely -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 2.2 MB/s eta 0:00:00


In [2]:
import osmnx as ox
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import pandas as pd

ox.settings.log_console = False
# Correct configuration for newer OSMnx versions to avoid auto-subdividing small queries
ox.settings.max_query_area_size = 100000000000

# Study area: Esplanade / Park Street / Maidan / BBD Bagh / Sealdah
north, south, east, west = 22.580, 22.550, 88.375, 88.340

# Free public Overpass servers sometimes refuse connections when busy.
# Try a few mirrors in order until one works.
overpass_mirrors = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]

G = None
roads_gdf = None
buildings_gdf = None
last_error = None
for mirror in overpass_mirrors:
    try:
        ox.settings.overpass_endpoint = mirror
        print(f"Trying {mirror} ...")
        print("Downloading real road network from OpenStreetMap...")
        G = ox.graph_from_bbox(bbox=(north, south, east, west), network_type="drive")
        roads_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True)

        print("Downloading real building footprints from OpenStreetMap...")
        buildings_gdf = ox.features_from_bbox(bbox=(north, south, east, west), tags={"building": True})
        buildings_gdf = buildings_gdf[buildings_gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]
        print(f"Success using {mirror}")
        break
    except Exception as e:
        last_error = e
        print(f"  Failed on {mirror}: {e}\n  Trying next mirror...\n")
        G = None
        continue

if G is None:
    raise RuntimeError(
        f"All Overpass mirrors failed. Last error: {last_error}\n"
        "Wait a few minutes and run this cell again (Overpass servers get overloaded "
        "and usually recover quickly) -- no need to re-run the pip install cell above."
    )

roads_m = roads_gdf.to_crs(epsg=32645)
buildings_m = buildings_gdf.to_crs(epsg=32645)

print(f"Got {len(roads_m)} road segments and {len(buildings_m)} buildings.")

Trying https://overpass-api.de/api/interpreter ...


/usr/local/lib/python3.13/dist-packages/osmnx/_overpass.py:271: UserWarning: This area is 2,509 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


  Failed on https://overpass-api.de/api/interpreter: 'overpass-api.de' responded: 502 Bad Gateway <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.22.1</center>
</body>
</html>

  Trying next mirror...

Trying https://overpass.kumi.systems/api/interpreter ...


/usr/local/lib/python3.13/dist-packages/osmnx/_overpass.py:271: UserWarning: This area is 2,509 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


  Failed on https://overpass.kumi.systems/api/interpreter: 'overpass-api.de' responded: 502 Bad Gateway <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.22.1</center>
</body>
</html>

  Trying next mirror...

Trying https://overpass.openstreetmap.ru/api/interpreter ...


/usr/local/lib/python3.13/dist-packages/osmnx/_overpass.py:271: UserWarning: This area is 2,509 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


  Failed on https://overpass.openstreetmap.ru/api/interpreter: 'overpass-api.de' responded: 502 Bad Gateway <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.22.1</center>
</body>
</html>

  Trying next mirror...

Trying https://overpass.private.coffee/api/interpreter ...


/usr/local/lib/python3.13/dist-packages/osmnx/_overpass.py:271: UserWarning: This area is 2,509 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


  Failed on https://overpass.private.coffee/api/interpreter: 'overpass-api.de' responded: 502 Bad Gateway <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.22.1</center>
</body>
</html>

  Trying next mirror...



RuntimeError: All Overpass mirrors failed. Last error: 'overpass-api.de' responded: 502 Bad Gateway <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.22.1</center>
</body>
</html>

Wait a few minutes and run this cell again (Overpass servers get overloaded and usually recover quickly) -- no need to re-run the pip install cell above.

In [ ]:
# Build 500m x 500m fishnet grid over the study area
minx, miny, maxx, maxy = roads_m.total_bounds
cell_size = 500  # meters
cols = int(np.ceil((maxx - minx) / cell_size))
rows = int(np.ceil((maxy - miny) / cell_size))

grid_cells = []
for i in range(cols):
    for j in range(rows):
        x0 = minx + i * cell_size
        y0 = miny + j * cell_size
        grid_cells.append({
            "grid_id": f"R{j}C{i}",
            "geometry": box(x0, y0, x0 + cell_size, y0 + cell_size)
        })

grid = gpd.GeoDataFrame(grid_cells, crs=roads_m.crs)
print(f"Built a {cols} x {rows} grid = {len(grid)} cells.")

In [ ]:
def box_counting_dimension(geom_list, cell_bounds, box_sizes_fraction=(0.5, 0.25, 0.125, 0.0625)):
    """
    geom_list: list of shapely geometries (roads or buildings) clipped to this cell
    cell_bounds: (minx, miny, maxx, maxy) of the 500m cell
    box_sizes_fraction: sub-box sizes as a fraction of the cell (2x2, 4x4, 8x8, 16x16 subdivisions)
    Returns fractal dimension = slope of log(N(eps)) vs log(1/eps)
    """
    if not geom_list:
        return np.nan

    minx, miny, maxx, maxy = cell_bounds
    width = maxx - minx
    height = maxy - miny

    log_inv_eps = []
    log_n = []

    for frac in box_sizes_fraction:
        box_w = width * frac
        box_h = height * frac
        n_boxes_x = max(1, int(round(1 / frac)))
        n_boxes_y = max(1, int(round(1 / frac)))

        occupied = 0
        for bi in range(n_boxes_x):
            for bj in range(n_boxes_y):
                sub_box = box(minx + bi * box_w, miny + bj * box_h,
                               minx + (bi + 1) * box_w, miny + (bj + 1) * box_h)
                if any(g.intersects(sub_box) for g in geom_list):
                    occupied += 1

        if occupied > 0:
            log_inv_eps.append(np.log(1 / frac))
            log_n.append(np.log(occupied))

    if len(log_n) < 2:
        return np.nan

    slope, _ = np.polyfit(log_inv_eps, log_n, 1)
    return round(slope, 4)

In [ ]:
results = []
print(f"Computing RFD/BFD for {len(grid)} grid cells (this is the slow part)...")
for idx, cell in grid.iterrows():
    cell_geom = cell.geometry
    bounds = cell_geom.bounds

    roads_in_cell = roads_m[roads_m.intersects(cell_geom)]
    road_geoms = [g.intersection(cell_geom) for g in roads_in_cell.geometry if not g.intersection(cell_geom).is_empty]

    buildings_in_cell = buildings_m[buildings_m.intersects(cell_geom)]
    building_geoms = [g.intersection(cell_geom) for g in buildings_in_cell.geometry if not g.intersection(cell_geom).is_empty]

    rfd = box_counting_dimension(road_geoms, bounds)
    bfd = box_counting_dimension(building_geoms, bounds)

    centroid_latlong = gpd.GeoSeries([cell_geom.centroid], crs=grid.crs).to_crs(epsg=4326).iloc[0]

    results.append({
        "grid_id": cell["grid_id"],
        "lat": round(centroid_latlong.y, 6),
        "long": round(centroid_latlong.x, 6),
        "RFD": rfd,
        "BFD": bfd,
        "n_road_segments": len(road_geoms),
        "n_buildings": len(building_geoms),
    })

out = pd.DataFrame(results)
out_clean = out.dropna(subset=["RFD"]).reset_index(drop=True)

out_clean.to_csv("kolkata_fractal_grid.csv", index=False)
print(f"\nDone. {len(out_clean)} grid cells with real computed RFD/BFD.")
out_clean.head(15)

**Next step:** click the folder icon on the left sidebar, find `kolkata_fractal_grid.csv`, right-click it, choose Download. Then upload that CSV to Claude.